# 03b — EIA-930 BA Interchange Flow Graph

**Purpose:** Pull hourly balancing authority interchange data from the EIA-930
API, aggregate it into a directed BA-to-BA flow graph, and export it for
visualization and analysis.

EIA-930 is the 'metabolism signal' of the US grid: it records, for every hour,
how many MWh flowed between each pair of neighbouring balancing authorities.
Positive values mean net export, negative means net import.

This notebook builds a NetworkX directed graph where:
- **Nodes** = balancing authorities (with capacity attributes from 03a)
- **Edges** = directed interchange corridors (with total and mean MWh flow)

**Inputs:**
- EIA-930 API (pulled here, no manual download)
- `data/processed/ba_territories.geojson` — from notebook 03a (for node geometry)
- `.env` with `EIA_API_KEY`

**Outputs:**
- `data/processed/eia930_raw.parquet` — raw hourly interchange records
- `data/processed/ba_interchange_summary.csv` — aggregated BA-pair flow totals
- `data/processed/grid_flow_network.graphml` — directed NetworkX graph
- `data/processed/flow_map.html` — Folium map of interchange edges

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import geopandas as gpd
import networkx as nx
import folium
from tqdm import tqdm
import utils

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# EIA Grid Monitor publishes bulk six-month CSVs — much faster than paginating
# the API (which times out on a full year of ~3 M records).
# Source: https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/

BULK_CSV_URLS = [
    'https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/EIA930_INTERCHANGE_2023_Jan_Jun.csv',
    'https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/EIA930_INTERCHANGE_2023_Jul_Dec.csv',
]

RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'eia930'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f'Bulk CSV files to download: {len(BULK_CSV_URLS)}')

## 1. Download EIA-930 Interchange Data (Bulk CSV)

EIA Grid Monitor publishes six-month bulk CSVs at:
`https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/`

Each file is ~96 MB and covers one half-year of hourly BA-to-BA interchange
(MW per hour = MWh for hourly intervals). We download both halves of 2023,
concatenate, and rename columns to the standard `fromba / toba / value / period` schema.

CSV columns: `Balancing Authority`, `Directly Interconnected Balancing Authority`,
`Interchange (MW)`, `UTC Time at End of Hour`, `Data Date`, `Hour Number`, etc.

In [ ]:
import requests

# ── Download bulk CSVs (skip if already cached) ───────────────────────────────
local_files = []
for url in BULK_CSV_URLS:
    fname = RAW_DIR / url.split('/')[-1]
    local_files.append(fname)
    if fname.exists():
        print(f'Already cached: {fname.name}')
        continue
    print(f'Downloading {fname.name} …', end=' ', flush=True)
    r = requests.get(url, timeout=300, stream=True)
    r.raise_for_status()
    with open(fname, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    print(f'{fname.stat().st_size / 1024 / 1024:.1f} MB')

print('\nAll files ready.')

In [ ]:
# ── Load and concatenate into DataFrame ───────────────────────────────────────
frames = [pd.read_csv(f) for f in local_files]
df930 = pd.concat(frames, ignore_index=True)
print('Shape:', df930.shape)
print('\nColumns:', df930.columns.tolist())
print('\nFirst record:')
print(df930.iloc[0].to_dict())

In [ ]:
# ── Rename columns to standard schema ────────────────────────────────���────────
# Bulk CSV uses verbose names; rename to match the rest of the notebook.
df930 = df930.rename(columns={
    'Balancing Authority':                       'fromba',
    'Directly Interconnected Balancing Authority': 'toba',
    'Interchange (MW)':                           'value',
    'UTC Time at End of Hour':                    'period',
})

df930['period'] = pd.to_datetime(df930['period'], utc=True, errors='coerce')
df930['value']  = pd.to_numeric(df930['value'], errors='coerce')

print('Date range:', df930['period'].min(), '→', df930['period'].max())
print('Null values:', df930['value'].isna().sum())
print('\nUnique FROM BAs:', df930['fromba'].nunique())
print('Unique TO BAs:  ', df930['toba'].nunique())

In [ ]:
# ── Save raw parquet ──────────────────────────────────────────────────────────
parquet_path = PROJECT_ROOT / 'data' / 'processed' / 'eia930_raw.parquet'
df930.to_parquet(parquet_path, index=False)
print(f'Saved raw data → {parquet_path}')

## 2. Aggregate: BA-Pair Flow Summary

Collapse the hourly records to a summary per directed BA pair:
- `total_mwh` — net MWh over the full period (positive = net exporter)
- `mean_mw` — mean hourly interchange (a proxy for typical corridor load)
- `hours_active` — count of non-null hours (data completeness check)

Note: EIA-930 reports both directions of each corridor separately
(A→B and B→A). We keep both directed edges — this captures the
asymmetry that matters for metabolism analysis.

In [ ]:
# ── Aggregate by directed BA pair ─────────────────────────────────────────────
flow_summary = (
    df930
    .dropna(subset=['value'])
    .groupby(['fromba', 'toba'])['value']
    .agg(
        total_mwh='sum',
        mean_mw='mean',
        hours_active='count'
    )
    .reset_index()
    .sort_values('total_mwh', ascending=False)
)

print(f'Unique directed BA pairs: {len(flow_summary):,}')
print('\nTop 15 corridors by total MWh exported:')
flow_summary.head(15)

In [ ]:
# ── Save summary CSV ──────────────────────────────────────────────────────────
csv_path = PROJECT_ROOT / 'data' / 'processed' / 'ba_interchange_summary.csv'
flow_summary.to_csv(csv_path, index=False)
print(f'Saved → {csv_path}')

## 3. Build NetworkX Directed Graph

Construct a `DiGraph` where:
- **Nodes** = BA codes, attributed with geometry centroid (lon/lat),
  BA name, and total installed capacity from 03a
- **Edges** = directed interchange corridors, weighted by `total_mwh`
  and `mean_mw`

In [ ]:
# ── Load BA territory centroids for node coordinates ─────────────────────────
ba_path = PROJECT_ROOT / 'data' / 'processed' / 'ba_territories.geojson'
ba = gpd.read_file(ba_path)

# Compute polygon centroids (project to equal-area first for accuracy)
ba_proj = ba.to_crs(epsg=5070)  # Conus Albers
ba['centroid_lon'] = ba_proj.geometry.centroid.to_crs(epsg=4326).x
ba['centroid_lat'] = ba_proj.geometry.centroid.to_crs(epsg=4326).y

# Build lookup dict: ba_code → node attributes
ba_attrs = ba.set_index('ba_code')[['ba_name', 'centroid_lon', 'centroid_lat', 'total_mw']].to_dict('index')
print(f'BA lookup entries: {len(ba_attrs)}')

In [ ]:
# ── Construct directed graph ──────────────────────────────────────────────────
G = nx.DiGraph()

# Add nodes — union of all BA codes seen in the flow data
all_ba_codes = set(flow_summary['fromba']) | set(flow_summary['toba'])
for code in all_ba_codes:
    attrs = ba_attrs.get(code, {})
    G.add_node(
        code,
        ba_name=attrs.get('ba_name', code),
        lon=attrs.get('centroid_lon', None),
        lat=attrs.get('centroid_lat', None),
        total_mw=float(attrs.get('total_mw', 0)),
    )

# Add directed edges
for _, row in flow_summary.iterrows():
    G.add_edge(
        row['fromba'],
        row['toba'],
        total_mwh=float(row['total_mwh']),
        mean_mw=float(row['mean_mw']),
        hours_active=int(row['hours_active']),
    )

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Weakly connected components: {nx.number_weakly_connected_components(G)}')

In [ ]:
# ── Graph diagnostics ─────────────────────────────────────────────────────────
degrees = dict(G.degree())
isolated = [n for n, d in degrees.items() if d == 0]
print(f'Isolated nodes (no edges): {len(isolated)}')
if isolated:
    print(' Isolated BAs:', isolated)

print('\nTop 10 nodes by total degree (most-connected BAs):')
top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:10]
for code, deg in top_nodes:
    name = G.nodes[code].get('ba_name', code)
    print(f'  {code:12s} ({name[:30]:<30}) degree={deg}')

In [ ]:
# ── Export GraphML ────────────────────────────────────────────────────────────
# GraphML requires all attributes to be scalar (str/int/float).
# None values will cause write errors — replace with empty string or 0.
for node, attrs in G.nodes(data=True):
    for k, v in attrs.items():
        if v is None:
            G.nodes[node][k] = ''

graphml_path = PROJECT_ROOT / 'data' / 'processed' / 'grid_flow_network.graphml'
nx.write_graphml(G, str(graphml_path))
print(f'Saved → {graphml_path}')

## 4. Folium Flow Map

Visualise the interchange graph on a map:
- BA territory polygons filled by net export position (exporter vs importer)
- Edges drawn as straight lines between BA centroids, scaled by `mean_mw`
- Arrow direction shows net flow direction

In [ ]:
# ── Compute net export position per BA ────────────────────────────────────────
# Sum all outgoing MWh minus all incoming MWh → positive = net exporter
exports = flow_summary.groupby('fromba')['total_mwh'].sum().rename('out_mwh')
imports = flow_summary.groupby('toba')['total_mwh'].sum().rename('in_mwh')
net_pos = pd.concat([exports, imports], axis=1).fillna(0)
net_pos['net_export_mwh'] = net_pos['out_mwh'] - net_pos['in_mwh']

ba = ba.merge(net_pos[['net_export_mwh']], left_on='ba_code', right_index=True, how='left')
ba['net_export_mwh'] = ba['net_export_mwh'].fillna(0)

print('Net exporters (top 5):')
print(ba.nlargest(5, 'net_export_mwh')[['ba_code', 'ba_name', 'net_export_mwh']].to_string(index=False))
print('\nNet importers (top 5):')
print(ba.nsmallest(5, 'net_export_mwh')[['ba_code', 'ba_name', 'net_export_mwh']].to_string(index=False))

In [ ]:
# ── Build flow map ────────────────────────────────────────────────────────────
import numpy as np

m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles='CartoDB positron')

# BA polygons — diverging fill: orange=exporter, blue=importer
max_abs = ba['net_export_mwh'].abs().quantile(0.95)

def net_color(val):
    norm = max(min(val / max_abs, 1), -1)
    if norm >= 0:
        intensity = int(60 + 195 * norm)
        return f'#{intensity:02x}6030'
    else:
        intensity = int(60 + 195 * (-norm))
        return f'#3060{intensity:02x}'

for _, row in ba.iterrows():
    if row.geometry is None:
        continue
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda _, v=row['net_export_mwh']: {
            'fillColor': net_color(v),
            'color': '#888888',
            'weight': 0.4,
            'fillOpacity': 0.5,
        },
        tooltip=f"{row.get('ba_code', '')} — {row.get('ba_name', '')}<br>"
                f"Net export: {row['net_export_mwh']:,.0f} MWh"
    ).add_to(m)

# Flow edges — lines between centroids, weight by mean_mw
# Only draw the top corridors by mean_mw to avoid clutter
TOP_N_EDGES = 80
top_edges = flow_summary.nlargest(TOP_N_EDGES, 'mean_mw')
max_mw = top_edges['mean_mw'].max()

for _, edge in top_edges.iterrows():
    src = G.nodes.get(edge['fromba'], {})
    dst = G.nodes.get(edge['toba'], {})
    if not src.get('lat') or not dst.get('lat'):
        continue
    width = 1 + 5 * (edge['mean_mw'] / max_mw)
    folium.PolyLine(
        locations=[
            [src['lat'], src['lon']],
            [dst['lat'], dst['lon']]
        ],
        color='#ff8800',
        weight=width,
        opacity=0.6,
        tooltip=f"{edge['fromba']} → {edge['toba']}: {edge['mean_mw']:,.0f} MW avg"
    ).add_to(m)

map_path = PROJECT_ROOT / 'data' / 'processed' / 'flow_map.html'
m.save(str(map_path))
print(f'Map saved → {map_path}')